In [3]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. User configuration
# ============================================================

# node2bin 文件，例如：
# {0: "chr1:0", 1: "chr1:1000000", ...}
NODE2BIN_FILE = (
    "hg38.1Mb.node2bin.npy"
)

# 输出目录
OUTDIR = Path(
    "GM12878_1Mb_feature_tables"
)
OUTDIR.mkdir(
    parents=True,
    exist_ok=True
)

# 1-Mb bin size
BIN_SIZE = 1_000_000

# 是否自动给染色体名称添加 chr 前缀
# 如果输入文件中已经是 chr1、chr2，则保持 False
ADD_CHR_PREFIX = False


# ------------------------------------------------------------
# Signal BED files
# ------------------------------------------------------------
#
# score_col 使用 Python 的 0-based 列号：
#
# narrowPeak:
#   chrom start end name score strand signalValue pValue qValue peak
#   signalValue 位于第 7 列，Python index = 6
#
# broadPeak:
#   通常使用 score 列，Python index = 4
#
# bedGraph:
#   chrom start end value
#   value 位于 Python index = 3
#
# 如果 BED 文件没有可靠的 signal score：
#   score_col = None
#   mode = "coverage_fraction"
#
# mode 可选：
#   weighted_mean
#       sum(signal * overlap_bp) / bin_size
#
#   weighted_overlap_mean
#       sum(signal * overlap_bp) / total_overlap_bp
#
#   score_sum
#       所有重叠区间 signal score 的总和
#
#   score_mean
#       所有重叠区间 signal score 的平均值
#
#   peak_count
#       与 bin 重叠的 peak 数量
#
#   coverage_bp
#       所有重叠区间的 overlap bp
#
#   coverage_fraction
#       coverage_bp / bin_size
#

SIGNAL_CONFIG = {
    "ATAC-seq": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/ATAC-seq/bed_peak/GM12878.ATAC-seq.ENCFF748UZH.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "H3K27ac": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K27ac.ENCFF367KIF.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "H3K4me1": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K4me1.ENCFF453PEP.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "H3K4me3": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K4me3.ENCFF188SZS.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "H3K27me3": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K27me3.ENCFF035PQG.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "CTCF": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/TF-ChIP-seq/bed_peak/GM12878.CTCF.ENCFF797SDL.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
    "SMC3": {
        "path": "/data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/TF-ChIP-seq/bed_peak/GM12878.SMC3.ENCFF837YJA.GRCh38.narrowPeak.bed",
        "score_col": None,
        "mode": "peak_count",
    },
}


# ------------------------------------------------------------
# Gene BED file
# ------------------------------------------------------------

GENE_BED_FILE = (
    "/data/xujs/References/human/hg38/hg38.gene.bed"
)

# 如果 gene BED 的第 4 列是 gene ID 或 gene symbol：
# BED 第4列对应 Python index = 3
#
# 如果没有 gene ID，设置为 None。
# None 时将统计重叠的 gene BED interval 数量。
GENE_ID_COL = 3

# ------------------------------------------------------------
# Super-enhancer BED file
# ------------------------------------------------------------

SUPER_ENHANCER_BED_FILE = (
    "/data/xujs/References/human/hg38/SuperEnhancer/GM12878.SE.hg38.bed"
)


In [4]:
# ============================================================
# 2. Utility functions
# ============================================================

def normalize_chromosome(chrom):
    """
    Standardize chromosome names.

    Examples:
        1    -> chr1, if ADD_CHR_PREFIX=True
        chr1 -> chr1
    """

    chrom = str(chrom)

    if ADD_CHR_PREFIX and not chrom.startswith("chr"):
        chrom = f"chr{chrom}"

    return chrom


def load_node2bin(
    node2bin_file,
    bin_size=1_000_000
):
    """
    Convert node2bin dictionary into a bin-level table.

    Input:
        {0: "chr1:0", 1: "chr1:1000000", ...}

    Output columns:
        node_id, chrom, start, end, center
    """

    node2bin = np.load(
        node2bin_file,
        allow_pickle=True
    ).item()

    records = []

    for node_id, region in node2bin.items():

        region = str(region)

        if ":" not in region:
            raise ValueError(
                f"Invalid node2bin region: {region}"
            )

        chrom, start = region.split(":", 1)

        chrom = normalize_chromosome(chrom)
        start = int(start)
        end = start + bin_size

        records.append({
            "node_id": int(node_id),
            "chrom": chrom,
            "start": start,
            "end": end,
            "center": (start + end) / 2.0,
        })

    bins = pd.DataFrame(records)

    bins = (
        bins
        .sort_values("node_id")
        .reset_index(drop=True)
    )

    return bins


def read_bed_file(path):
    """
    Read BED-like files robustly.

    Supported:
        BED3
        BED4
        BED6
        narrowPeak
        broadPeak
        bedGraph

    Columns are retained as:
        col0, col1, col2, ...

    Standard columns:
        chrom, start, end
    """

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"File does not exist: {path}"
        )

    rows = []

    with path.open("r") as f:

        for raw_line in f:

            line = raw_line.strip()

            if not line:
                continue

            if line.startswith("#"):
                continue

            if line.startswith("track"):
                continue

            if line.startswith("browser"):
                continue

            fields = line.split()

            if len(fields) < 3:
                continue

            # Skip possible header lines
            first = fields[0].lower()
            second = fields[1].lower()

            if first in {
                "chrom",
                "chromosome"
            }:
                continue

            if second == "start":
                continue

            rows.append(fields)

    if len(rows) == 0:
        raise ValueError(
            f"No valid BED records found in: {path}"
        )

    max_ncol = max(
        len(row)
        for row in rows
    )

    padded_rows = [
        row + [None] * (max_ncol - len(row))
        for row in rows
    ]

    raw = pd.DataFrame(
        padded_rows,
        columns=[
            f"col{i}"
            for i in range(max_ncol)
        ]
    )

    raw["chrom"] = (
        raw["col0"]
        .map(normalize_chromosome)
    )

    raw["start"] = pd.to_numeric(
        raw["col1"],
        errors="coerce"
    )

    raw["end"] = pd.to_numeric(
        raw["col2"],
        errors="coerce"
    )

    raw = raw.dropna(
        subset=[
            "chrom",
            "start",
            "end"
        ]
    ).copy()

    raw["start"] = (
        raw["start"]
        .astype(np.int64)
    )

    raw["end"] = (
        raw["end"]
        .astype(np.int64)
    )

    # BED 使用 0-based、half-open 坐标
    raw = raw[
        raw["end"] > raw["start"]
    ].copy()

    raw = (
        raw
        .sort_values(
            ["chrom", "start", "end"]
        )
        .reset_index(drop=True)
    )

    return raw


def add_signal_column(
    interval_df,
    score_col=None
):
    """
    Add numeric signal column.

    score_col:
        0-based raw BED column index.

    If score_col is None:
        every interval receives signal = 1.
    """

    df = interval_df.copy()

    if score_col is None:

        df["signal"] = 1.0

    else:

        raw_col = f"col{score_col}"

        if raw_col not in df.columns:
            raise ValueError(
                f"{raw_col} not found. "
                f"Available columns: {list(df.columns)}"
            )

        df["signal"] = pd.to_numeric(
            df[raw_col],
            errors="coerce"
        ).fillna(0.0)

    return df


def aggregate_interval_feature(
    bins,
    intervals,
    feature_name,
    mode="weighted_mean"
):
    """
    Aggregate interval-based signals into genomic bins.

    Returns:
        feature value
        overlap peak count
        overlap bp
    """

    if len(intervals) == 0:
        return pd.DataFrame({
            feature_name: np.zeros(len(bins)),
            f"{feature_name}_peak_count": np.zeros(
                len(bins),
                dtype=np.int64
            ),
            f"{feature_name}_overlap_bp": np.zeros(
                len(bins),
                dtype=np.int64
            ),
        })

    grouped_intervals = {}

    for chrom, sub_df in intervals.groupby(
        "chrom",
        sort=False
    ):

        sub_df = (
            sub_df
            .sort_values("start")
            .reset_index(drop=True)
        )

        grouped_intervals[chrom] = {
            "start": sub_df["start"].to_numpy(
                dtype=np.int64
            ),
            "end": sub_df["end"].to_numpy(
                dtype=np.int64
            ),
            "signal": sub_df["signal"].to_numpy(
                dtype=float
            ),
        }

    feature_values = np.zeros(
        len(bins),
        dtype=float
    )

    peak_counts = np.zeros(
        len(bins),
        dtype=np.int64
    )

    overlap_bp_values = np.zeros(
        len(bins),
        dtype=np.int64
    )

    for i, row in enumerate(
        bins.itertuples(index=False)
    ):

        chrom = row.chrom

        if chrom not in grouped_intervals:
            continue

        grouped = grouped_intervals[chrom]

        starts = grouped["start"]
        ends = grouped["end"]
        signals = grouped["signal"]

        bin_start = int(row.start)
        bin_end = int(row.end)
        bin_length = bin_end - bin_start

        # 只有 start < bin_end 的 intervals 才可能重叠
        right = np.searchsorted(
            starts,
            bin_end,
            side="left"
        )

        if right == 0:
            continue

        candidate_indices = np.arange(right)

        # 进一步要求 interval_end > bin_start
        keep = ends[:right] > bin_start

        candidate_indices = (
            candidate_indices[keep]
        )

        if len(candidate_indices) == 0:
            continue

        interval_starts = starts[
            candidate_indices
        ]

        interval_ends = ends[
            candidate_indices
        ]

        interval_signals = signals[
            candidate_indices
        ]

        overlap_starts = np.maximum(
            interval_starts,
            bin_start
        )

        overlap_ends = np.minimum(
            interval_ends,
            bin_end
        )

        overlap_lengths = (
            overlap_ends - overlap_starts
        )

        valid = overlap_lengths > 0

        overlap_lengths = (
            overlap_lengths[valid]
        )

        interval_signals = (
            interval_signals[valid]
        )

        if len(overlap_lengths) == 0:
            continue

        n_peaks = len(overlap_lengths)
        total_overlap_bp = int(
            overlap_lengths.sum()
        )

        weighted_signal_sum = float(
            np.sum(
                interval_signals
                * overlap_lengths
            )
        )

        peak_counts[i] = n_peaks
        overlap_bp_values[i] = total_overlap_bp

        if mode == "weighted_mean":

            # signal * overlap length / bin length
            feature_values[i] = (
                weighted_signal_sum
                / bin_length
            )

        elif mode == "weighted_overlap_mean":

            feature_values[i] = (
                weighted_signal_sum
                / total_overlap_bp
            )

        elif mode == "score_sum":

            feature_values[i] = (
                interval_signals.sum()
            )

        elif mode == "score_mean":

            feature_values[i] = (
                interval_signals.mean()
            )

        elif mode == "peak_count":

            feature_values[i] = float(
                n_peaks
            )

        elif mode == "coverage_bp":

            feature_values[i] = float(
                total_overlap_bp
            )

        elif mode == "coverage_fraction":

            feature_values[i] = (
                total_overlap_bp
                / bin_length
            )

        else:

            raise ValueError(
                f"Unknown aggregation mode: {mode}"
            )

    return pd.DataFrame({
        feature_name: feature_values,
        f"{feature_name}_peak_count": peak_counts,
        f"{feature_name}_overlap_bp": overlap_bp_values,
    })


def aggregate_unique_id_count(
    bins,
    intervals,
    id_col=None,
    feature_name="gene_count"
):
    """
    Count unique IDs overlapping each genomic bin.

    id_col:
        Raw BED column index containing gene ID/symbol.

    If id_col is None:
        count overlapping intervals instead.
    """

    df = intervals.copy()

    if id_col is None:

        df["_interval_id"] = [
            f"interval_{i}"
            for i in range(len(df))
        ]

    else:

        raw_col = f"col{id_col}"

        if raw_col not in df.columns:
            raise ValueError(
                f"{raw_col} not found in gene BED file."
            )

        df["_interval_id"] = (
            df[raw_col]
            .fillna("")
            .astype(str)
        )

        # 空 ID 使用 interval index 替代
        empty_mask = (
            df["_interval_id"]
            .isin(["", ".", "None", "nan"])
        )

        df.loc[
            empty_mask,
            "_interval_id"
        ] = [
            f"interval_{i}"
            for i in df.index[empty_mask]
        ]

    grouped_intervals = {}

    for chrom, sub_df in df.groupby(
        "chrom",
        sort=False
    ):

        sub_df = (
            sub_df
            .sort_values("start")
            .reset_index(drop=True)
        )

        grouped_intervals[chrom] = {
            "start": sub_df["start"].to_numpy(
                dtype=np.int64
            ),
            "end": sub_df["end"].to_numpy(
                dtype=np.int64
            ),
            "ids": sub_df["_interval_id"].to_numpy(
                dtype=str
            ),
        }

    gene_counts = np.zeros(
        len(bins),
        dtype=np.int64
    )

    for i, row in enumerate(
        bins.itertuples(index=False)
    ):

        chrom = row.chrom

        if chrom not in grouped_intervals:
            continue

        grouped = grouped_intervals[chrom]

        starts = grouped["start"]
        ends = grouped["end"]
        ids = grouped["ids"]

        bin_start = int(row.start)
        bin_end = int(row.end)

        right = np.searchsorted(
            starts,
            bin_end,
            side="left"
        )

        if right == 0:
            continue

        candidate_indices = np.arange(right)

        keep = ends[:right] > bin_start

        candidate_indices = (
            candidate_indices[keep]
        )

        if len(candidate_indices) == 0:
            continue

        gene_counts[i] = len(
            np.unique(
                ids[candidate_indices]
            )
        )

    return pd.DataFrame({
        feature_name: gene_counts
    })


In [5]:
# ============================================================
# 3. Create the 1-Mb genomic-bin table
# ============================================================

bins = load_node2bin(
    NODE2BIN_FILE,
    bin_size=BIN_SIZE
)

print(
    f"Number of 1-Mb bins: {len(bins)}"
)


# ============================================================
# 4. Aggregate epigenomic signals
# ============================================================

feature_df = bins.copy()

qc_df = bins[
    [
        "node_id",
        "chrom",
        "start",
        "end"
    ]
].copy()

for feature_name, config in SIGNAL_CONFIG.items():

    path = config["path"]
    score_col = config.get(
        "score_col",
        None
    )
    mode = config.get(
        "mode",
        "weighted_mean"
    )

    print(
        f"Processing {feature_name}: {path}"
    )

    signal_bed = read_bed_file(path)

    signal_bed = add_signal_column(
        signal_bed,
        score_col=score_col
    )

    aggregated = aggregate_interval_feature(
        bins=bins,
        intervals=signal_bed,
        feature_name=feature_name,
        mode=mode
    )

    feature_df[feature_name] = (
        aggregated[feature_name]
        .to_numpy()
    )

    qc_df[
        f"{feature_name}_peak_count"
    ] = aggregated[
        f"{feature_name}_peak_count"
    ].to_numpy()

    qc_df[
        f"{feature_name}_overlap_bp"
    ] = aggregated[
        f"{feature_name}_overlap_bp"
    ].to_numpy()

    print(
        f"  non-zero bins: "
        f"{np.sum(feature_df[feature_name] > 0)}"
    )


# ============================================================
# 5. Aggregate gene counts
# ============================================================

print(
    f"Processing genes: {GENE_BED_FILE}"
)

gene_bed = read_bed_file(
    GENE_BED_FILE
)

gene_count_df = aggregate_unique_id_count(
    bins=bins,
    intervals=gene_bed,
    id_col=GENE_ID_COL,
    feature_name="gene_count"
)

feature_df["gene_count"] = (
    gene_count_df["gene_count"]
    .to_numpy()
)


# ============================================================
# 6. Aggregate super-enhancers
# ============================================================

print(
    f"Processing super-enhancers: "
    f"{SUPER_ENHANCER_BED_FILE}"
)

super_enhancer_bed = read_bed_file(
    SUPER_ENHANCER_BED_FILE
)

# Super-enhancer BED 如果没有可靠的 signal score，
# 为每个区间赋 signal = 1
super_enhancer_bed = add_signal_column(
    super_enhancer_bed,
    score_col=None
)

super_enhancer_agg = (
    aggregate_interval_feature(
        bins=bins,
        intervals=super_enhancer_bed,
        feature_name="super_enhancer_overlap_bp",
        mode="coverage_bp"
    )
)

feature_df["super_enhancer_overlap_bp"] = (
    super_enhancer_agg[
        "super_enhancer_overlap_bp"
    ].to_numpy()
)

feature_df["super_enhancer"] = (
    feature_df[
        "super_enhancer_overlap_bp"
    ] > 0
).astype(np.int8)

feature_df["super_enhancer_count"] = (
    super_enhancer_agg[
        "super_enhancer_overlap_bp_peak_count"
    ].to_numpy()
)

qc_df["super_enhancer_overlap_bp"] = (
    feature_df[
        "super_enhancer_overlap_bp"
    ]
)

qc_df["super_enhancer_count"] = (
    feature_df[
        "super_enhancer_count"
    ]
)


# ============================================================
# 7. Reorder and save raw feature table
# ============================================================

signal_feature_names = list(
    SIGNAL_CONFIG.keys()
)

feature_columns = [
    "node_id",
    "chrom",
    "start",
    "end",
] + signal_feature_names + [
    "gene_count",
    "super_enhancer",
    "super_enhancer_overlap_bp",
    "super_enhancer_count",
]

feature_df = feature_df[
    feature_columns
].copy()

# 数值列缺失值填 0
numeric_columns = [
    c for c in feature_df.columns
    if c not in {
        "node_id",
        "chrom",
    }
]

feature_df[numeric_columns] = (
    feature_df[numeric_columns]
    .fillna(0)
)

raw_output = (
    OUTDIR
    / "GM12878_1Mb_bin_features_raw.tsv"
)

feature_df.to_csv(
    raw_output,
    sep="\t",
    index=False
)

print(
    f"Raw feature table saved to: {raw_output}"
)


# ============================================================
# 8. Optional normalization
# ============================================================

# 对连续特征进行 log1p + z-score。
# 适用于不同信号量纲差异较大的情况。
#
# 注意：
# 1. 如果后续模型在训练集上进行标准化，
#    更严格的做法是只用训练染色体估计均值和标准差。
# 2. 这里的全基因组 z-score 主要用于探索性分析和可视化。

normalized_df = feature_df.copy()

continuous_features = (
    signal_feature_names
    + [
        "gene_count",
        "super_enhancer_overlap_bp",
        "super_enhancer_count",
    ]
)

normalization_statistics = []

for col in continuous_features:

    values = pd.to_numeric(
        normalized_df[col],
        errors="coerce"
    ).fillna(0.0)

    log_values = np.log1p(values)

    mean_value = log_values.mean()
    std_value = log_values.std(ddof=0)

    if std_value == 0:
        normalized_values = np.zeros(
            len(log_values),
            dtype=float
        )
    else:
        normalized_values = (
            log_values - mean_value
        ) / std_value

    normalized_df[col] = normalized_values

    normalization_statistics.append({
        "feature": col,
        "mean_log1p": mean_value,
        "sd_log1p": std_value,
    })

normalization_statistics_df = pd.DataFrame(
    normalization_statistics
)

normalized_output = (
    OUTDIR
    / "GM12878_1Mb_bin_features_log1p_zscore.tsv"
)

normalized_df.to_csv(
    normalized_output,
    sep="\t",
    index=False
)

normalization_statistics_df.to_csv(
    OUTDIR
    / "feature_normalization_statistics.tsv",
    sep="\t",
    index=False
)

print(
    f"Normalized feature table saved to: "
    f"{normalized_output}"
)


# ============================================================
# 9. Save QC table
# ============================================================

qc_output = (
    OUTDIR
    / "GM12878_1Mb_bin_features_QC.tsv"
)

qc_df.to_csv(
    qc_output,
    sep="\t",
    index=False
)

print(
    f"QC table saved to: {qc_output}"
)


# ============================================================
# 10. Print summary
# ============================================================

print("\nFeature summary:")

for col in signal_feature_names:

    print(
        f"{col}: "
        f"min={feature_df[col].min():.4f}, "
        f"median={feature_df[col].median():.4f}, "
        f"max={feature_df[col].max():.4f}, "
        f"nonzero="
        f"{np.sum(feature_df[col] > 0)}"
    )

print(
    f"gene_count: "
    f"min={feature_df['gene_count'].min()}, "
    f"median={feature_df['gene_count'].median()}, "
    f"max={feature_df['gene_count'].max()}"
)

print(
    f"super_enhancer bins: "
    f"{feature_df['super_enhancer'].sum()}"
)

print("\nDone.")

Number of 1-Mb bins: 3044
Processing ATAC-seq: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/ATAC-seq/bed_peak/GM12878.ATAC-seq.ENCFF748UZH.GRCh38.narrowPeak.bed
  non-zero bins: 2852
Processing H3K27ac: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K27ac.ENCFF367KIF.GRCh38.narrowPeak.bed
  non-zero bins: 2421
Processing H3K4me1: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K4me1.ENCFF453PEP.GRCh38.narrowPeak.bed
  non-zero bins: 2411
Processing H3K4me3: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K4me3.ENCFF188SZS.GRCh38.narrowPeak.bed
  non-zero bins: 2380
Processing H3K27me3: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/Histone-ChIP-seq/bed_peak/GM12878.H3K27me3.ENCFF035PQG.GRCh38.narrowPeak.bed
  non-zero bins: 2089
Processing CTCF: /data/xujs/Public_Data/ENCODE_Data/human/hg38/GM12878/TF-ChIP-seq/bed_peak/GM12878.CTCF.ENCFF797SDL.G